# Notebook 6. Giải vòng tròn cấp 1–7 và tổng hợp

Người phụ trách: Nguyễn Trọng Phúc.

Nội dung:
1. Giải vòng tròn 7 cấp, mỗi cặp `tournament.games_per_pair` ván, đổi bên đi trước sau mỗi ván, mỗi ván một seed.
2. Ma trận đối đầu, bảng thắng/hòa/thua, khoảng tin cậy Wilson 95%, xếp hạng.
3. Ảnh hưởng bên đi trước.
4. Chi phí: thời gian mỗi nước (TB, p95), số nút/mô phỏng, kích thước mô hình, thời gian huấn luyện, bộ nhớ.
5. Độ chính xác chiến thuật của cả 7 cấp.
6. Ván minh họa: một ván cấp mạnh hơn thua, một ván bình thường (lấy theo quy tắc cố định, không chọn tay).

Cấu hình mặc định của từng cấp lấy từ `config.yaml`. Mỗi ván mở đầu bằng 2 nước ngẫu nhiên có seed để các tác tử tất định (cấp 5, 6) không lặp lại một ván.
Cần có đủ mô hình trong `files/` (nb3, nb4, nb5) và `files/tactics.json` (nb2).

In [ ]:
# QUICK = True: chạy thử vài ván, vài epoch để kiểm tra code (ghi vào thư mục con thu_nhanh/)
# QUICK = False: chạy thật theo config.yaml, kết quả dùng cho báo cáo
QUICK = False

import os
if not os.path.exists("config.yaml"):
    raise RuntimeError("Hãy mở Jupyter trong thư mục final_connect_four (nơi có config.yaml).")

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.experiments import setup, summaries_table, with_random_opening, move_times_of
from utils.arena import run_match, format_summary, read_log, first_player_stats
from utils.data import machine_info
from utils.conn_simple_env import conn
from utils import tactics
from utils.arena import round_robin
from utils.experiments import plot_matrix, matrix_table, info_mean, records_of, LEVEL_LABELS
ex = setup(QUICK)
cfg = ex.cfg
print("Cấu hình máy:", machine_info())

## 1. Giải vòng tròn

In [ ]:
tour_cfg = cfg["tournament"]
names = tour_cfg["agents"]
opening = tour_cfg["opening_random_moves"]
for name in names:
    if not ex.model_ready(name):
        raise RuntimeError(f"Thiếu mô hình cho {name}, chạy nb3, nb4, nb5 trước.")

agents = {}
for name in names:
    agents[name] = with_random_opening(ex.agent(name), opening)

n_games = ex.n(tour_cfg["games_per_pair"], 4)
log_path = ex.log("nb6_giai_dau.jsonl")
start = time.perf_counter()
summaries, matrix = round_robin(agents, n_games, seed=tour_cfg["seed"], log_path=log_path)
print("Giải đấu chạy mất", round(time.perf_counter() - start), "giây")

tour_table = summaries_table(summaries)
tour_table.to_csv(ex.table("nb6_giai_dau.csv"), index=False, encoding="utf-8-sig")
tour_table

In [ ]:
matrix_df = matrix_table(matrix, names)
matrix_df.to_csv(ex.table("nb6_ma_tran.csv"), encoding="utf-8-sig")
plot_matrix(matrix, names, ex.fig("nb6_ma_tran.png"))
matrix_df

In [ ]:
# Xếp hạng theo score trung bình khi gặp 6 cấp còn lại
rows = []
for a in names:
    scores = []
    for b in names:
        if a != b:
            scores.append(matrix[a][b])
    rows.append({"tác tử": LEVEL_LABELS[a], "score TB": round(float(np.mean(scores)), 3)})
rank_table = pd.DataFrame(rows).sort_values("score TB", ascending=False).reset_index(drop=True)
rank_table.index = rank_table.index + 1
rank_table.to_csv(ex.table("nb6_xep_hang.csv"), encoding="utf-8-sig")
rank_table

## 2. Ảnh hưởng bên đi trước

Connect Four 7×6 đã được giải: bên đi trước thắng nếu chơi hoàn hảo. Bảng dưới cho tỷ lệ chung và score của mỗi tác tử khi cầm đỏ so với khi cầm vàng.

In [ ]:
records = read_log(log_path)
print("Toàn giải:", first_player_stats(records))

rows = []
for a in names:
    as_red = []
    as_yellow = []
    for s in summaries:
        if s["agent"] == a:
            as_red.append(s["score_as_red"])
            as_yellow.append(s["score_as_yellow"])
        if s["opponent"] == a:
            # thống kê lưu theo góc agent nên đổi sang góc của a
            as_red.append(1 - s["score_as_yellow"])
            as_yellow.append(1 - s["score_as_red"])
    rows.append({"tác tử": LEVEL_LABELS[a], "score TB khi cầm đỏ": round(float(np.mean(as_red)), 3),
                 "score TB khi cầm vàng": round(float(np.mean(as_yellow)), 3)})
first_table = pd.DataFrame(rows)
first_table.to_csv(ex.table("nb6_ben_di_truoc.csv"), index=False, encoding="utf-8-sig")
first_table

## 3. Chi phí tính toán

Thời gian mỗi nước tính trên mọi ván của giải, bỏ 2 nước khai cuộc ngẫu nhiên. Cấp 3 ghi số nút, cấp 4 và 7 ghi số mô phỏng.

In [ ]:
def all_records_of(agent_name):
    result = []
    for r in records:
        if r["red"] == agent_name or r["yellow"] == agent_name:
            result.append(r)
    return result

def file_size_mb(path):
    if os.path.exists(path):
        return round(os.path.getsize(path) / 1e6, 2)
    return None

def train_seconds(path):
    meta_path = path + ".json"
    if not os.path.exists(meta_path):
        return None
    with open(meta_path, encoding="utf-8") as f:
        meta = json.load(f)
    report = meta.get("report", {})
    if "train_seconds" in report:
        return report["train_seconds"]
    total = 0
    for stage in ["stage1", "stage2"]:
        if stage in report:
            total += report[stage]["seconds"]
    return total

model_files = {
    "cap5": [ex.model("policy_conn.keras")],
    "cap6": [ex.model("pg_conn.keras")],
    "cap7": [ex.model("policy_conn.keras"), ex.model("value_conn.keras")],
}

rows = []
for name in names:
    agent_records = all_records_of(name)
    mean_t, p95_t = move_times_of(agent_records, name)
    row = {"tác tử": LEVEL_LABELS[name], "TB (ms)": round(mean_t * 1000, 1), "p95 (ms)": round(p95_t * 1000, 1),
           "số nút TB": info_mean(agent_records, name, "nodes"),
           "số mô phỏng TB": info_mean(agent_records, name, "simulations")}
    if name in model_files:
        sizes = [file_size_mb(p) for p in model_files[name]]
        row["mô hình (MB)"] = sum(x for x in sizes if x is not None)
        row["huấn luyện (giây)"] = sum(train_seconds(p) or 0 for p in model_files[name])
    rows.append(row)
cost_table = pd.DataFrame(rows)
cost_table.to_csv(ex.table("nb6_chi_phi.csv"), index=False, encoding="utf-8-sig")
cost_table

In [ ]:
# Bộ nhớ của tiến trình notebook sau khi đã nạp đủ 7 tác tử (cần thư viện psutil)
try:
    import psutil
    rss = psutil.Process(os.getpid()).memory_info().rss / 1e6
    print(f"RAM tiến trình: {rss:.0f} MB")
except ImportError:
    print("Chưa cài psutil (pip install psutil) nên không đo được RAM.")

## 4. Độ chính xác chiến thuật của 7 cấp

In [ ]:
positions = tactics.load_tactics(ex.model("tactics.json"))
rows = []
for name in names:
    move_fn, info_fn = ex.agent(name)
    row = {"tác tử": LEVEL_LABELS[name]}
    row.update(tactics.score_agent(move_fn, positions))
    rows.append(row)
tactics_table = pd.DataFrame(rows).round(3)
tactics_table.to_csv(ex.table("nb6_chien_thuat.csv"), index=False, encoding="utf-8-sig")
tactics_table

In [ ]:
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, 4.2))
width = 0.25
for offset, group, label in [(-width, "win_now", "thắng ngay"), (0, "must_block", "buộc chặn"),
                             (width, "double_threat", "tạo đòn đôi")]:
    ax.bar(x + offset, tactics_table[group], width=width, label=label)
ax.set_xticks(x, [LEVEL_LABELS[n] for n in names])
ax.set_ylim(0, 1.05)
ax.set_ylabel("tỷ lệ chọn đúng")
ax.set_title("Độ chính xác chiến thuật theo nhóm thế cờ")
ax.legend()
fig.tight_layout()
fig.savefig(ex.fig("nb6_chien_thuat.png"), dpi=150)

## 5. Ván minh họa

Quy tắc chọn cố định để tránh chọn riêng ván đẹp:
- ván đầu tiên trong log mà cấp có score trung bình cao nhất bị thua;
- ván đầu tiên trong log giữa hai cấp đứng đầu bảng xếp hạng.

In [ ]:
code_by_label = {}
for name in names:
    code_by_label[LEVEL_LABELS[name]] = name
top = code_by_label[rank_table.iloc[0]["tác tử"]]
second = code_by_label[rank_table.iloc[1]["tác tử"]]

def show_game(record, title):
    env = conn.from_moves(record["moves"])
    print(title)
    print(f"Đỏ: {record['red']}, vàng: {record['yellow']}, người thắng: {record['winner']}, số nước: {len(record['moves'])}")
    print("Các nước:", record["moves"])
    print(env.render())
    print()

loss_game = None
for r in records:
    loser_is_top = (r["winner"] == "red" and r["yellow"] == top) or (r["winner"] == "yellow" and r["red"] == top)
    if loser_is_top:
        loss_game = r
        break
if loss_game is None:
    print(LEVEL_LABELS[top], "không thua ván nào trong giải.")
else:
    show_game(loss_game, f"Ván {LEVEL_LABELS[top]} thua:")

for r in records:
    if set([r["red"], r["yellow"]]) == set([top, second]):
        show_game(r, f"Ván giữa hai cấp đứng đầu ({LEVEL_LABELS[top]} và {LEVEL_LABELS[second]}):")
        break

In [ ]:
# Thế cờ chiến thuật mà cấp đứng đầu chọn sai (nếu có)
move_fn, info_fn = ex.agent(top)
wrong = 0
for i, pos in enumerate(positions):
    import random
    random.seed(i)
    np.random.seed(i)
    env = conn.from_moves(pos["moves"])
    move = move_fn(env.clone())
    if move not in pos["answers"]:
        wrong += 1
        if wrong <= 3:
            print(f"{pos['id']}: {LEVEL_LABELS[top]} đi cột {move}, đáp án {pos['answers']}")
            print(env.render())
            print()
print("Tổng số thế chọn sai:", wrong, "/", len(positions))